# Analysis — prompting shots (0 / 1 / 3 examples)

Does a fine-tuned model do better when the prompt also contains training examples? Every model is compared with
0, 1 and 3 examples in the prompt (1 shot = the first of the 3 few-shot examples).

| Model | 0 shots | 1 shot | 3 shots |
|---|---|---|---|
| vanilla 7B | `vanilla` | `vanilla-1shot` | `few-shot` |
| Qwen 14B | `qwen-14b` | `qwen-14b-1shot` | `qwen-14b-3shot` |
| LoRA (seeds 42, 43, 44) | `lora/seed-*` | `lora-1shot/seed-*` | `lora-3shot/seed-*` |
| QLoRA (seeds 42, 43, 44) | `qlora/seed-*` | `qlora-1shot/seed-*` | `qlora-3shot/seed-*` |
| LoRA hyperparameter variants (seed 42) | `hp-lora-<c>/seed-42` | `hp-lora-<c>-1shot/seed-42` | `hp-lora-<c>-3shot/seed-42` |

Same units, seed aggregation and tests as `experiment/experiment/analysis/index.ipynb`: structural metrics per unique
test instruction (median of duplicate rows, median of seeds), DS-1000 Pass@1 / Pass@10 as binary values per problem
(majority of seeds); Friedman / Cochran's Q, then Wilcoxon / McNemar with Holm correction and paired bootstrap CIs.
Cells whose runs are not on the Hub yet are skipped. Structural results need `code4ml_ratings.csv`
(`experiment/experiment/rating/rate_outputs.ipynb`).

In [ ]:
%pip install -q pandas scipy statsmodels huggingface_hub matplotlib   # into the selected kernel (Colab or local)

import os

try:  # Colab secret; elsewhere an existing `hf auth login` is used
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

## 1. Runs and helpers

In [ ]:
import itertools
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
from scipy import stats
from statsmodels.stats.contingency_tables import cochrans_q, mcnemar
from statsmodels.stats.multitest import multipletests

pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 30)

REPO = 'dimsmauls/code4ml-eval-outputs'
SEEDS = (42, 43, 44)
SHOTS = (0, 1, 3)
ALPHA = 0.05
BOOTSTRAP = 10_000
SEED = 42

# model -> ({shots: run name}, seeds); seeds None = one run without a seed folder
MODELS = {
    'vanilla': ({0: 'vanilla', 1: 'vanilla-1shot', 3: 'few-shot'}, None),
    'qwen-14b': ({0: 'qwen-14b', 1: 'qwen-14b-1shot', 3: 'qwen-14b-3shot'}, None),
    'lora': ({0: 'lora', 1: 'lora-1shot', 3: 'lora-3shot'}, SEEDS),
    'qlora': ({0: 'qlora', 1: 'qlora-1shot', 3: 'qlora-3shot'}, SEEDS),
    **{f'hp-lora-{c}': ({0: f'hp-lora-{c}', 1: f'hp-lora-{c}-1shot', 3: f'hp-lora-{c}-3shot'}, (42,))
       for c in ('cosine', 'r32', 'lr1e-4')},
}
CELLS = {(model, k): ([name] if seeds is None else [f'{name}/seed-{s}' for s in seeds])
         for model, (names, seeds) in MODELS.items() for k, name in names.items()}

files = set(HfApi().list_repo_files(REPO, repo_type='dataset'))
available = {cell: runs for cell, runs in CELLS.items() if all(f'{r}/summary.json' in files for r in runs)}
rated = {cell: runs for cell, runs in available.items() if all(f'{r}/code4ml_ratings.csv' in files for r in runs)}
print('cells on the Hub:', len(available), 'of', len(CELLS), '| rated:', len(rated))
print('missing:', sorted(set(CELLS) - set(available)) or 'none')
print('not rated yet:', sorted(set(available) - set(rated)) or 'none')


def label(cell):
    return f'{cell[0]} | {cell[1]}-shot'


def get(path):
    return hf_hub_download(REPO, path, repo_type='dataset')


# ---- same helpers as experiment/experiment/analysis/index.ipynb ----
def holm(pvalues):
    return multipletests(pvalues, method='holm')[1]


def bootstrap_ci(diffs):
    '''Paired bootstrap over units: 95% CI of the mean of per-unit differences.'''
    diffs = np.asarray(diffs, dtype=float)
    rng = np.random.default_rng(SEED)
    means = diffs[rng.integers(0, len(diffs), (BOOTSTRAP, len(diffs)))].mean(axis=1)
    return np.percentile(means, [2.5, 97.5])


def friedman(table, columns):
    block = table[columns].dropna()
    chi2, p = stats.friedmanchisquare(*(block[c] for c in columns))
    return {'n': len(block), 'chi2': chi2, 'p': p, 'kendall_w': chi2 / (len(block) * (len(columns) - 1))}


def wilcoxon_family(table, pairs):
    rows = []
    for a, b in pairs:
        diff = (table[a] - table[b]).dropna()
        nonzero = diff[diff != 0]
        if len(nonzero):
            p = stats.wilcoxon(nonzero).pvalue
            ranks = stats.rankdata(nonzero.abs())
            r = (ranks[nonzero.to_numpy() > 0].sum() - ranks[nonzero.to_numpy() < 0].sum()) / ranks.sum()
        else:
            p, r = 1.0, 0.0
        rows.append({'comparison': f'{a} − {b}', 'n': len(diff), 'median_diff': diff.median(),
                     'mean_diff': diff.mean(), 'p': p, 'rank_biserial': r})
    out = pd.DataFrame(rows)
    out['p_holm'] = holm(out['p'])
    out['significant'] = out['p_holm'] < ALPHA
    return out


def mcnemar_family(table, pairs):
    rows = []
    for a, b in pairs:
        block = table[[a, b]].dropna().astype(int)
        both, only_a = ((block[a] == 1) & (block[b] == 1)).sum(), ((block[a] == 1) & (block[b] == 0)).sum()
        only_b, neither = ((block[a] == 0) & (block[b] == 1)).sum(), ((block[a] == 0) & (block[b] == 0)).sum()
        p = mcnemar([[both, only_a], [only_b, neither]], exact=True).pvalue
        low, high = bootstrap_ci(block[a] - block[b])
        rows.append({'comparison': f'{a} − {b}', 'n': len(block), 'rate_a': block[a].mean(), 'rate_b': block[b].mean(),
                     'diff': block[a].mean() - block[b].mean(), 'ci95_low': low, 'ci95_high': high,
                     'a_only': only_a, 'b_only': only_b, 'p': p})
    out = pd.DataFrame(rows)
    out['p_holm'] = holm(out['p'])
    out['significant'] = out['p_holm'] < ALPHA
    return out


def combine(frames, how):
    '''One value per unit for a cell: median of the seeds, only units present in every seed.'''
    if len(frames) == 1:
        return frames[0]
    stacked = pd.concat(frames, keys=range(len(frames)))
    median = stacked.groupby(level=1).median()
    complete = stacked.groupby(level=1).count() == len(frames)
    return how(median.where(complete))

## 2. Load ratings and DS-1000 results

In [ ]:
GRADE = {g: i for i, g in enumerate('ABCDE', 1)}
STRUCTURAL = ['maintainability', 'code_smells', 'complexity', 'cognitive_complexity', 'bugs', 'mlscent_ml_rating',
              'code_smells_per_100_ncloc', 'has_import']


def has_import(code):
    return float(any(line.lstrip().startswith(('import ', 'from ')) for line in str(code).splitlines()))


def load_ratings(run):
    '''Same as analysis/index.ipynb (SonarQube N/A without ncloc), plus has_import (1 = the answer imports something).'''
    df = pd.read_csv(get(f'{run}/code4ml_ratings.csv'), dtype={'id': str})
    valid = df['ast_valid'].astype(str) == 'True'
    ncloc = pd.to_numeric(df['ncloc'], errors='coerce')
    sonar = valid & (ncloc > 0)
    out = pd.DataFrame(index=df['id'])
    out['ast_valid'] = valid.to_numpy()
    out['ncloc'] = ncloc.where(sonar).to_numpy()
    out['maintainability'] = df['maintainability_rating'].map(GRADE).where(sonar).to_numpy()
    for metric in ('code_smells', 'complexity', 'cognitive_complexity', 'bugs'):
        out[metric] = pd.to_numeric(df[metric], errors='coerce').where(sonar).to_numpy()
    out['mlscent_ml_rating'] = df['mlscent_ml_rating'].map(GRADE).where(valid).to_numpy()
    out['code_smells_per_100_ncloc'] = 100 * out['code_smells'] / out['ncloc']
    out['has_import'] = df['code_block'].map(has_import).where(valid).to_numpy()
    return out


# unit of the structural tests = unique test instruction (304 rows share 40 instruction texts)
INSTRUCTION_OF = pd.read_csv(get('vanilla/code4ml_ratings.csv'), dtype={'id': str}).set_index('id')['graph_vertex_subclass']


def by_instruction(df):
    return df.drop(columns='ast_valid').groupby(INSTRUCTION_OF.reindex(df.index).to_numpy()).median()


def ds1000_binary(run):
    res = pd.read_json(get(f'{run}/ds1000_results.jsonl'), lines=True)
    first = res[res['sample'] == 0].set_index('problem_id')['passed'].astype(float)
    any_pass = res.groupby('problem_id')['passed'].any().astype(float)
    return pd.DataFrame({'pass1': first, 'pass10': any_pass})


ratings = {r: load_ratings(r) for runs in rated.values() for r in runs}
struct = {cell: combine([by_instruction(ratings[r]) for r in runs], lambda df: df) for cell, runs in rated.items()}
binary = {cell: combine([ds1000_binary(r) for r in runs], lambda df: df.round()) for cell, runs in available.items()}
summaries = {r: json.load(open(get(f'{r}/summary.json'))) for runs in available.values() for r in runs}
print('loaded', len(summaries), 'runs')

## 3. Matrices (model × shots)
Pass@k: unbiased estimate, mean of seeds. Structural: mean over unique instructions (LoRA / QLoRA: median of seeds
first). `has_import`: share of instructions whose answer imports something.

In [ ]:
def matrix(value, cells):
    return pd.DataFrame({f'{k}-shot': {m: value((m, k)) if (m, k) in cells else np.nan for m in MODELS}
                         for k in SHOTS})


for k in ('pass@1', 'pass@10'):
    print(f'\n=== DS-1000 {k}')
    display(matrix(lambda cell: np.mean([summaries[r]['ds1000'][k] for r in available[cell]]), available).round(4))

for metric in ['ncloc'] + STRUCTURAL:
    how = (lambda cell: struct[cell][metric].median()) if metric == 'ncloc' else (lambda cell: struct[cell][metric].mean())
    print(f"\n=== {'median' if metric == 'ncloc' else 'mean'} {metric}")
    display(matrix(how, struct).round(3))

In [ ]:
# Pass@k per seed (descriptive only)
pd.DataFrame([{'cell': label(cell), 'run': r, 'pass@1': summaries[r]['ds1000']['pass@1'],
               'pass@10': summaries[r]['ds1000']['pass@10']}
              for cell, runs in available.items() if len(runs) > 1 for r in runs]).set_index(['cell', 'run']).round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, k in zip(axes, ('pass@1', 'pass@10')):
    for model in MODELS:
        points = [(s, np.mean([summaries[r]['ds1000'][k] for r in available[(model, s)]]))
                  for s in SHOTS if (model, s) in available]
        if points:
            ax.plot(*zip(*points), marker='o', linewidth=1.5, label=model)
    ax.set_xticks(SHOTS, [f'{s}-shot' for s in SHOTS])
    ax.set_title(f'DS-1000 {k}', loc='left', fontsize=11)
    ax.grid(alpha=0.3)
axes[0].set_ylabel('DS-1000 (997 problems)')
axes[1].legend(frameon=False, fontsize=8, loc='lower right')
plt.tight_layout()
plt.show()

## 4. Within each model: 0 vs 1 vs 3 shots
Cochran's Q (DS-1000) / Friedman (structural) over the available shot levels, then McNemar / Wilcoxon for every pair
of shot levels with Holm correction inside the model. With only two shot levels the pair is tested directly.

In [ ]:
def across_shots(model):
    print(f'\n########## {model} ##########')
    cells = [(model, s) for s in SHOTS if (model, s) in available]
    if len(cells) < 2:
        print('fewer than two shot levels on the Hub: skipped')
        return
    for k in ('pass1', 'pass10'):
        table = pd.DataFrame({label(c): binary[c][k] for c in cells})
        if len(cells) >= 3:
            block = table.dropna()
            q = cochrans_q(block.to_numpy())
            print(f"=== {k} | Cochran's Q n={len(block)} Q={q.statistic:.3f} p={q.pvalue:.4g}")
            if q.pvalue >= ALPHA:
                print('not significant: no post-hoc')
                continue
        display(mcnemar_family(table, list(itertools.combinations(table.columns, 2))).round(4))
    cells = [c for c in cells if c in struct]
    if len(cells) < 2:
        print('structural: fewer than two rated shot levels, skipped')
        return
    for metric in STRUCTURAL:
        table = pd.DataFrame({label(c): struct[c][metric] for c in cells})
        if table.dropna().nunique().max() <= 1:
            print(f'=== {metric}: all values equal, nothing to test')
            continue
        if len(cells) >= 3:
            omnibus = friedman(table, list(table.columns))
            print(f"=== {metric} | Friedman n={omnibus['n']} chi2={omnibus['chi2']:.3f} p={omnibus['p']:.4g} "
                  f"Kendall's W={omnibus['kendall_w']:.3f}")
            if omnibus['p'] >= ALPHA:
                continue
        else:
            print(f'=== {metric}')
        display(wilcoxon_family(table, list(itertools.combinations(table.columns, 2))).round(4))


for model in MODELS:
    across_shots(model)

## 5. Within each shot level: models compared
Same families as the main analysis: LoRA / QLoRA against vanilla and against each other, and Qwen 14B against LoRA,
QLoRA and vanilla 7B (separate Holm family). At 3 shots, vanilla is the existing few-shot arm, so this answers whether
fine-tuning plus examples beats examples alone.

In [ ]:
FAMILIES = {'main': [('lora', 'vanilla'), ('qlora', 'vanilla'), ('lora', 'qlora')],
            '14b': [('qwen-14b', 'lora'), ('qwen-14b', 'qlora'), ('qwen-14b', 'vanilla')]}

for s in SHOTS:
    for family, model_pairs in FAMILIES.items():
        pairs = [(label((a, s)), label((b, s))) for a, b in model_pairs if (a, s) in available and (b, s) in available]
        if not pairs:
            continue
        print(f'\n########## {s}-shot, family {family} ##########')
        for k in ('pass1', 'pass10'):
            table = pd.DataFrame({label(c): binary[c][k] for c in available if c[1] == s})
            print(f'=== {k}')
            display(mcnemar_family(table, pairs).round(4))
        rated_pairs = [(a, b) for a, b in pairs if all(c in map(label, struct) for c in (a, b))]
        for metric in STRUCTURAL if rated_pairs else []:
            table = pd.DataFrame({label(c): struct[c][metric] for c in struct if c[1] == s})
            print(f'=== {metric}')
            display(wilcoxon_family(table, rated_pairs).round(4))